# Face Detection Localization Evaluation (added in revision)

This notebook does **not** retrain any model. It re-evaluates the saved deblurred outputs of U-Net, ViT and ViT-U-Net (the same files used in `Thesis_Face_Detection_Rate_Evaluation.ipynb`) and adds:

1. Reference face boxes from the sharp test images (MTCNN, same settings as dataset filtering).
2. Localization-based evaluation: a detection is correct only if IoU ≥ 0.5 with a reference box → TP / FP / FN, precision, recall, FP per image.
3. The original detection rate (≥1 face, confidence > 0.9) recomputed as a sanity check against Table 3.
4. Sample counts and 95% Wilson confidence intervals per cell, plus a Sharp (upper bound) row.
5. Exact McNemar tests (paired, same test images) with Holm correction.

In [1]:
!pip install -q deepface opencv-contrib-python gdown
!pip install -q facenet-pytorch --no-deps

In [2]:
# Same Google Drive files used in Evaluation/Thesis_Face_Detection_Rate_Evaluation.ipynb
!gdown -q https://drive.google.com/uc?id=1zgtvkRJb3y9XhaS1uu9QvYvKxZmmmrGx -O /content/deblurred_outputs_unet.zip
!gdown -q https://drive.google.com/uc?id=1fjUJm5OpWcWnpHOQYiH8GXuMcLaShTxe -O /content/deblurred_outputs_vit.zip
!gdown -q https://drive.google.com/uc?id=1PDfH4iwk6tuc9G1aVuXTPzxEo7YJ1fdL -O /content/deblurred_outputs_vit-unet.zip
!gdown -q https://drive.google.com/uc?id=1V48xVqhC-D2DQbkvi4AEZrT_3mYWFl5T -O /content/Final_Modeling_Dataset.zip

!unzip -q -o /content/deblurred_outputs_unet.zip     -d /content/deblurred_outputs_unet
!unzip -q -o /content/deblurred_outputs_vit.zip      -d /content/deblurred_outputs_vit
!unzip -q -o /content/deblurred_outputs_vit-unet.zip -d /content/deblurred_outputs_vit-unet
!unzip -q -o /content/Final_Modeling_Dataset.zip     -d /content/Final_Modeling_Dataset

In [3]:
import os, csv, glob, urllib.request
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

TEST_BLUR_DIR  = "/content/Final_Modeling_Dataset/test/blur"
TEST_SHARP_DIR = "/content/Final_Modeling_Dataset/test/sharp"
RESULTS_DIR    = "/content/localization_eval"
os.makedirs(RESULTS_DIR, exist_ok=True)

LEVELS = ["low", "medium", "high", "very_high"]

def find_dir_with(root, filename):
    """Locate the folder under `root` that contains `filename` (robust to zip nesting)."""
    hits = glob.glob(os.path.join(root, "**", filename), recursive=True)
    if not hits:
        raise FileNotFoundError(f"{filename} not found under {root}")
    return os.path.dirname(hits[0])

# Output naming conventions taken from the training notebooks:
#   U-Net    -> deblur_{id}_{level}.jpg
#   ViT      -> {id}_{level}.jpg
#   ViT-UNet -> {id}_{level}.jpg
MODELS = {
    "U-Net":    {"dir": find_dir_with("/content/deblurred_outputs_unet", "deblur_0_low.jpg"), "prefix": "deblur_"},
    "ViT":      {"dir": find_dir_with("/content/deblurred_outputs_vit", "0_low.jpg"),         "prefix": ""},
    "ViT-U-Net":{"dir": find_dir_with("/content/deblurred_outputs_vit-unet", "0_low.jpg"),    "prefix": ""},
}
for k, v in MODELS.items():
    print(f"{k:<10} -> {v['dir']}")

# Detector settings — identical to the original evaluation notebook
RETINA_CONF = 0.9          # DeepFace RetinaFace: keep faces with confidence > 0.9
YUNET_CONF  = 0.9          # YuNet score threshold
YUNET_NMS   = 0.3
YUNET_MODEL = "/content/face_detection_yunet_2023mar.onnx"

# Ground truth (MTCNN on sharp test images) — same settings as dataset filtering
GT_CONF     = 0.90
IOU_THR     = 0.5
IOU_SWEEP   = [0.3, 0.5, 0.7]

# Image ids in the test split
ids = sorted({os.path.splitext(f)[0].rsplit("_low", 1)[0]
              for f in os.listdir(TEST_BLUR_DIR) if f.endswith("_low.jpg")}, key=int)
print(f"Test source images: {len(ids)}  (expected 156)")

U-Net      -> /content/deblurred_outputs_unet/output_images
ViT        -> /content/deblurred_outputs_vit/deblurred_outputs_vit
ViT-U-Net  -> /content/deblurred_outputs_vit-unet/deblurred_outputs
Test source images: 156  (expected 156)


## Helper functions (IoU, matching, Wilson CI, McNemar, Holm)

In [4]:
"""Pure helper functions for the localization evaluation (no detector deps)."""
import math
import numpy as np


def iou(a, b):
    """IoU of two boxes in (x1, y1, x2, y2) format."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def match_detections(dets, gts, thr=0.5):
    """Greedy one-to-one matching, highest-score detection first.

    dets : list of (x1, y1, x2, y2, score)
    gts  : list of (x1, y1, x2, y2)
    Returns (tp, fp, fn).
    """
    order = sorted(range(len(dets)), key=lambda i: -dets[i][4])
    used = set()
    tp = 0
    for i in order:
        best_j, best_iou = -1, thr
        for j, g in enumerate(gts):
            if j in used:
                continue
            v = iou(dets[i][:4], g)
            if v >= best_iou:
                best_j, best_iou = j, v
        if best_j >= 0:
            used.add(best_j)
            tp += 1
    fp = len(dets) - tp
    fn = len(gts) - tp
    return tp, fp, fn


def primary_iou(dets, primary_box):
    """Best IoU between any detection and the primary (largest) GT face."""
    if primary_box is None or not dets:
        return 0.0
    return max(iou(d[:4], primary_box) for d in dets)


def wilson_ci(k, n, z=1.959963984540054):
    """95% Wilson score interval, returned in percent."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h) * 100, min(1.0, c + h) * 100)


def mcnemar_exact(a, b):
    """Exact McNemar test on paired boolean arrays a, b.

    Returns (b_only_a, b_only_b, p_value). b_only_a = cases where a succeeds
    and b fails; b_only_b = the reverse.
    """
    from scipy.stats import binomtest
    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)
    n01 = int(np.sum(a & ~b))
    n10 = int(np.sum(~a & b))
    n = n01 + n10
    if n == 0:
        return n01, n10, 1.0
    p = binomtest(n01, n, 0.5, alternative="two-sided").pvalue
    return n01, n10, float(p)


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (same order as input)."""
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    adj = [0.0] * m
    running = 0.0
    for rank, i in enumerate(order):
        val = min(1.0, (m - rank) * pvals[i])
        running = max(running, val)
        adj[i] = running
    return adj

## Detectors
All detectors return a list of `(x1, y1, x2, y2, score)` for detections that pass the confidence threshold.

In [5]:
from deepface import DeepFace

def detect_retinaface(path):
    try:
        faces = DeepFace.extract_faces(img_path=path, detector_backend="retinaface",
                                       enforce_detection=False, align=False)
    except Exception:
        return []
    out = []
    for f in faces:
        conf = f.get("confidence", 0) or 0
        if conf > RETINA_CONF:          # with enforce_detection=False DeepFace returns the
            a = f["facial_area"]        # whole image with confidence 0 when nothing is found
            out.append((a["x"], a["y"], a["x"] + a["w"], a["y"] + a["h"], float(conf)))
    return out

if not os.path.exists(YUNET_MODEL):
    urllib.request.urlretrieve(
        "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
        YUNET_MODEL)
_yunet = cv2.FaceDetectorYN.create(YUNET_MODEL, "", (320, 320), YUNET_CONF, YUNET_NMS, 5000)

def detect_yunet(path):
    img = cv2.imread(path)
    if img is None:
        return []
    h, w = img.shape[:2]
    _yunet.setInputSize((w, h))
    _, faces = _yunet.detect(img)
    if faces is None:
        return []
    return [(float(f[0]), float(f[1]), float(f[0] + f[2]), float(f[1] + f[3]), float(f[14]))
            for f in faces]

DETECTORS = {"RetinaFace": detect_retinaface, "YuNet": detect_yunet}

26-11-02 14:20:30 - Directory /root/.deepface has been created
26-11-02 14:20:30 - Directory /root/.deepface/weights has been created


## Ground-truth boxes from sharp images (MTCNN)
Primary face = the largest detected face. All faces with probability ≥ 0.90 are used as ground truth for precision/recall.

In [6]:
import torch
from facenet_pytorch import MTCNN
mtcnn = MTCNN(keep_all=True, device="cuda" if torch.cuda.is_available() else "cpu",
              min_face_size=20, thresholds=[0.6, 0.7, 0.90], post_process=False)

GT = {}   # id -> {"boxes": [...], "primary": box or None}
rows = []
for i in tqdm(ids, desc="GT (MTCNN on sharp)"):
    img = cv2.cvtColor(cv2.imread(os.path.join(TEST_SHARP_DIR, f"{i}.jpg")), cv2.COLOR_BGR2RGB)
    boxes, probs = mtcnn.detect(img)
    keep = []
    if boxes is not None:
        keep = [tuple(map(float, b)) for b, p in zip(boxes, probs) if p >= GT_CONF]
    primary = max(keep, key=lambda b: (b[2] - b[0]) * (b[3] - b[1])) if keep else None
    GT[i] = {"boxes": keep, "primary": primary}
    for b in keep:
        rows.append({"id": i, "x1": b[0], "y1": b[1], "x2": b[2], "y2": b[3],
                     "is_primary": b == primary})
pd.DataFrame(rows).to_csv(f"{RESULTS_DIR}/gt_boxes.csv", index=False)

no_gt = [i for i in ids if GT[i]["primary"] is None]
multi = [i for i in ids if len(GT[i]["boxes"]) > 1]
print(f"Images with GT face : {len(ids) - len(no_gt)}/{len(ids)}")
print(f"Images without GT   : {len(no_gt)} -> excluded from localization metrics {no_gt}")
print(f"Images with >1 face : {len(multi)}")

GT (MTCNN on sharp): 100%|██████████| 156/156 [00:08<00:00, 17.64it/s]

Images with GT face : 155/156
Images without GT   : 1 -> excluded from localization metrics ['48']
Images with >1 face : 27


## Run detection on Sharp, Blurred, and the 3 restoration models

In [7]:
def image_path(condition, i, level):
    if condition == "Sharp":
        return os.path.join(TEST_SHARP_DIR, f"{i}.jpg")
    if condition == "Blurred":
        return os.path.join(TEST_BLUR_DIR, f"{i}_{level}.jpg")
    m = MODELS[condition]
    return os.path.join(m["dir"], f"{m['prefix']}{i}_{level}.jpg")

jobs = [("Sharp", "sharp")] + [(c, l) for c in ["Blurred"] + list(MODELS) for l in LEVELS]

per_image = []
for det_name, det_fn in DETECTORS.items():
    for cond, level in jobs:
        for i in tqdm(ids, desc=f"{det_name:<10} {cond:<10} {level:<9}", leave=False):
            p = image_path(cond, i, level)
            if not os.path.exists(p):
                raise FileNotFoundError(p)
            dets = det_fn(p)
            gt = GT[i]
            row = {"detector": det_name, "condition": cond, "level": level, "id": i,
                   "n_det": len(dets), "any_det": len(dets) > 0,
                   "has_gt": gt["primary"] is not None, "n_gt": len(gt["boxes"])}
            if row["has_gt"]:
                tp, fp, fn = match_detections(dets, gt["boxes"], IOU_THR)
                piou = primary_iou(dets, gt["primary"])
                row.update(tp=tp, fp=fp, fn=fn, primary_iou=piou)
                for t in IOU_SWEEP:
                    row[f"loc@{t}"] = piou >= t
            per_image.append(row)
        print(f"done: {det_name} / {cond} / {level}")

df = pd.DataFrame(per_image)
for t in IOU_SWEEP:   # rows without GT have NaN here; force a clean boolean dtype
    df[f"loc@{t}"] = df[f"loc@{t}"].astype("boolean").fillna(False).astype(bool)
df.to_csv(f"{RESULTS_DIR}/per_image_results.csv", index=False)
print(df.shape)

RetinaFace Sharp      sharp    :   0%|          | 0/156 [00:00<?, ?it/s]

26-11-02 14:21:00 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5

  0%|          | 0.00/119M [00:00<?, ?B/s]
 31%|███       | 36.7M/119M [00:00<00:00, 364MB/s]
 62%|██████▏   | 73.4M/119M [00:00<00:00, 362MB/s]
100%|██████████| 119M/119M [00:00<00:00, 133MB/s]


done: RetinaFace / Sharp / sharp


done: RetinaFace / Blurred / low


done: RetinaFace / Blurred / medium


... (27 outputs left out)

done: YuNet / ViT-U-Net / low


done: YuNet / ViT-U-Net / medium


done: YuNet / ViT-U-Net / high


done: YuNet / ViT-U-Net / very_high
(5304, 15)


## Sanity check: original detection rates must match Table 3 of the paper

In [15]:
PAPER_TABLE3 = {  # detected / 156 as reported in the paper
    ("RetinaFace", "Blurred"):   [156, 156, 97, 28],
    ("YuNet",      "Blurred"):   [151, 133, 32, 2],
    ("RetinaFace", "U-Net"):     [156, 156, 156, 155],
    ("YuNet",      "U-Net"):     [154, 153, 152, 148],
    ("RetinaFace", "ViT"):       [156, 156, 151, 138],
    ("YuNet",      "ViT"):       [150, 142, 123, 92],
    ("RetinaFace", "ViT-U-Net"): [156, 156, 156, 156],
    ("YuNet",      "ViT-U-Net"): [152, 152, 149, 149],
}
all_ok = True
for (det, cond), expected in PAPER_TABLE3.items():
    got = [int(df[(df.detector == det) & (df.condition == cond) & (df.level == l)].any_det.sum())
           for l in LEVELS]
    ok = got == expected
    all_ok &= ok
    print(f"{det:<10} {cond:<10} paper={expected} now={got} {'OK' if ok else '<-- DIFFERENT'}")
print("\nALL MATCH" if all_ok else
      "\nMismatch found: likely due to different DeepFace/OpenCV versions. Report the new numbers consistently.")

RetinaFace Blurred    paper=[156, 156, 97, 28] now=[156, 156, 97, 28] OK
YuNet      Blurred    paper=[151, 133, 32, 2] now=[151, 133, 32, 2] OK
RetinaFace U-Net      paper=[156, 156, 156, 155] now=[156, 156, 156, 155] OK
YuNet      U-Net      paper=[154, 153, 152, 148] now=[154, 153, 152, 148] OK
RetinaFace ViT        paper=[156, 156, 151, 138] now=[156, 156, 151, 138] OK
YuNet      ViT        paper=[150, 142, 123, 92] now=[150, 142, 123, 92] OK
RetinaFace ViT-U-Net  paper=[156, 156, 156, 156] now=[156, 156, 156, 156] OK
YuNet      ViT-U-Net  paper=[152, 152, 149, 149] now=[152, 152, 149, 149] OK

ALL MATCH


## Summary per detector x condition x blur level

In [9]:
def fmt_ci(k, n):
    lo, hi = wilson_ci(k, n)
    return f"{k}/{n} ({100*k/n:.1f}%) [{lo:.1f}–{hi:.1f}]"

order_cond  = ["Sharp", "Blurred"] + list(MODELS)
order_level = ["sharp"] + LEVELS

summary = []
for det in DETECTORS:
    for cond in order_cond:
        for level in order_level:
            g = df[(df.detector == det) & (df.condition == cond) & (df.level == level)]
            if g.empty:
                continue
            gg = g[g.has_gt]
            n, ng = len(g), len(gg)
            k_any = int(g.any_det.sum())
            k_loc = int(gg[f"loc@{IOU_THR}"].sum())
            tp, fp, fn = int(gg.tp.sum()), int(gg.fp.sum()), int(gg.fn.sum())
            mislocalized = int((gg.any_det & ~gg[f"loc@{IOU_THR}"]).sum())
            hits = gg[gg[f"loc@{IOU_THR}"]]
            summary.append({
                "detector": det, "condition": cond, "level": level, "n": n,
                "detected_any": k_any,
                "detection_rate_%": round(100 * k_any / n, 1),
                "det_CI95": "%.1f–%.1f" % wilson_ci(k_any, n),
                "n_with_gt": ng,
                "correct_loc": k_loc,
                "correct_loc_%": round(100 * k_loc / ng, 1),
                "loc_CI95": "%.1f–%.1f" % wilson_ci(k_loc, ng),
                "detected_but_mislocalized": mislocalized,
                "TP": tp, "FP": fp, "FN": fn,
                "precision": round(tp / (tp + fp), 4) if tp + fp else float("nan"),
                "recall": round(tp / (tp + fn), 4) if tp + fn else float("nan"),
                "FP_per_image": round(fp / ng, 3),
                "mean_IoU_of_hits": round(hits.primary_iou.mean(), 3) if len(hits) else float("nan"),
            })
summary = pd.DataFrame(summary)
summary.to_csv(f"{RESULTS_DIR}/summary.csv", index=False)
pd.set_option("display.width", 250, "display.max_columns", 30)
summary

,detector,condition,level,n,detected_any,detection_rate_%,det_CI95,n_with_gt,correct_loc,correct_loc_%,loc_CI95,detected_but_mislocalized,TP,FP,FN,precision,recall,FP_per_image,mean_IoU_of_hits
0,RetinaFace,Sharp,sharp,156,156,100.0,97.6–100.0,155,148,95.5,91.0–97.8,7,175,19,9,0.9021,0.9511,0.123,0.876
1,RetinaFace,Blurred,low,156,156,100.0,97.6–100.0,155,148,95.5,91.0–97.8,7,176,17,8,0.9119,0.9565,0.110,0.879
2,RetinaFace,Blurred,medium,156,156,100.0,97.6–100.0,155,148,95.5,91.0–97.8,7,166,10,18,0.9432,0.9022,0.065,0.832
3,RetinaFace,Blurred,high,156,97,62.2,54.4–69.4,155,90,58.1,50.2–65.5,7,100,3,84,0.9709,0.5435,0.019,0.754
4,RetinaFace,Blurred,very_high,156,28,17.9,12.7–24.7,155,24,15.5,10.6–22.0,4,24,4,160,0.8571,0.1304,0.026,0.658
5,RetinaFace,U-Net,low,156,156,100.0,97.6–100.0,155,148,95.5,91.0–97.8,7,175,14,9,0.9259,0.9511,0.090,0.879
6,RetinaFace,U-Net,medium,156,156,100.0,97.6–100.0,155,148,95.5,91.0–97.8,7,172,14,12,0.9247,0.9348,0.090,0.887
7,RetinaFace,U-Net,high,156,156,100.0,97.6–100.0,155,148,95.5,91.0–97.8,7,171,14,13,0.9243,0.9293,0.090,0.885
8,RetinaFace,U-Net,very_high,156,155,99.4,96.5–99.9,155,147,94.8,90.1–97.4,7,166,10,18,0.9432,0.9022,0.065,0.885
9,RetinaFace,ViT,low,156,156,100.0,97.6–100.0,155,148,95.5,91.0–97.8,7,173,14,11,0.9251,0.9402,0.090,0.884


## IoU threshold sensitivity (primary-face localization)

In [10]:
sweep = []
for det in DETECTORS:
    for cond in order_cond:
        for level in order_level:
            gg = df[(df.detector == det) & (df.condition == cond) & (df.level == level) & df.has_gt]
            if gg.empty:
                continue
            r = {"detector": det, "condition": cond, "level": level, "n": len(gg)}
            for t in IOU_SWEEP:
                r[f"IoU≥{t}"] = f"{100 * gg[f'loc@{t}'].mean():.1f}%"
            sweep.append(r)
sweep = pd.DataFrame(sweep)
sweep.to_csv(f"{RESULTS_DIR}/iou_sweep.csv", index=False)
sweep

,detector,condition,level,n,IoU≥0.3,IoU≥0.5,IoU≥0.7
0,RetinaFace,Sharp,sharp,155,96.8%,95.5%,93.5%
1,RetinaFace,Blurred,low,155,96.8%,95.5%,93.5%
2,RetinaFace,Blurred,medium,155,96.8%,95.5%,90.3%
3,RetinaFace,Blurred,high,155,58.1%,58.1%,41.9%
4,RetinaFace,Blurred,very_high,155,17.4%,15.5%,5.8%
5,RetinaFace,U-Net,low,155,96.8%,95.5%,94.8%
6,RetinaFace,U-Net,medium,155,96.8%,95.5%,94.2%
7,RetinaFace,U-Net,high,155,96.8%,95.5%,92.9%
8,RetinaFace,U-Net,very_high,155,96.1%,94.8%,93.5%
9,RetinaFace,ViT,low,155,96.8%,95.5%,92.9%


## Exact McNemar tests (paired) on correct localization, with Holm correction per detector

In [11]:
def outcome(det, cond, level, col):
    g = df[(df.detector == det) & (df.condition == cond) & (df.level == level) & df.has_gt]
    return g.sort_values("id")[col].values

pairs = [(m, "Blurred") for m in MODELS] + [("ViT-U-Net", "U-Net"), ("U-Net", "ViT"), ("ViT-U-Net", "ViT")]
tests = []
for det in DETECTORS:
    block = []
    for level in LEVELS:
        for a, b in pairs:
            n_a, n_b, p = mcnemar_exact(outcome(det, a, level, f"loc@{IOU_THR}"),
                                        outcome(det, b, level, f"loc@{IOU_THR}"))
            block.append({"detector": det, "level": level, "A": a, "B": b,
                          "A_only": n_a, "B_only": n_b, "p_raw": p})
    for r, padj in zip(block, holm([r["p_raw"] for r in block])):
        r["p_holm"] = padj
        r["significant_0.05"] = padj < 0.05
    tests += block
tests = pd.DataFrame(tests)
tests.to_csv(f"{RESULTS_DIR}/mcnemar.csv", index=False)
tests

,detector,level,A,B,A_only,B_only,p_raw,p_holm,significant_0.05
0,RetinaFace,low,U-Net,Blurred,0,0,1.000000e+00,1.000000e+00,False
1,RetinaFace,low,ViT,Blurred,0,0,1.000000e+00,1.000000e+00,False
2,RetinaFace,low,ViT-U-Net,Blurred,0,0,1.000000e+00,1.000000e+00,False
3,RetinaFace,low,ViT-U-Net,U-Net,0,0,1.000000e+00,1.000000e+00,False
4,RetinaFace,low,U-Net,ViT,0,0,1.000000e+00,1.000000e+00,False
5,RetinaFace,low,ViT-U-Net,ViT,0,0,1.000000e+00,1.000000e+00,False
6,RetinaFace,medium,U-Net,Blurred,0,0,1.000000e+00,1.000000e+00,False
7,RetinaFace,medium,ViT,Blurred,0,0,1.000000e+00,1.000000e+00,False
8,RetinaFace,medium,ViT-U-Net,Blurred,0,0,1.000000e+00,1.000000e+00,False
9,RetinaFace,medium,ViT-U-Net,U-Net,0,0,1.000000e+00,1.000000e+00,False


## Paper-ready tables (markdown)

In [12]:
lines = []
for title, col_k, col_n in [("Table 3 (revised): detection rate (>=1 face, conf > 0.9), k/n (%), 95% Wilson CI",
                             "detected_any", "n"),
                            (f"New table: correct localization (IoU >= {IOU_THR} with GT primary face), k/n (%), 95% Wilson CI",
                             "correct_loc", "n_with_gt")]:
    lines.append(f"### {title}\n")
    lines.append("| Condition | Detector | " + " | ".join(order_level) + " |")
    lines.append("|---|---|" + "---|" * len(order_level))
    for cond in order_cond:
        for det in DETECTORS:
            cells = []
            for level in order_level:
                r = summary[(summary.detector == det) & (summary.condition == cond) & (summary.level == level)]
                cells.append(fmt_ci(int(r[col_k].iloc[0]), int(r[col_n].iloc[0])) if len(r) else "–")
            lines.append(f"| {cond} | {det} | " + " | ".join(cells) + " |")
    lines.append("")

lines.append(f"### New table: precision / recall / FP per image (IoU >= {IOU_THR})\n")
lines.append("| Condition | Detector | Level | Precision | Recall | FP/img | Detected but mislocalized |")
lines.append("|---|---|---|---|---|---|---|")
for _, r in summary.iterrows():
    lines.append(f"| {r.condition} | {r.detector} | {r.level} | {r.precision:.3f} | {r.recall:.3f} | "
                 f"{r.FP_per_image:.3f} | {r.detected_but_mislocalized} |")

text = "\n".join(lines)
open(f"{RESULTS_DIR}/paper_tables.md", "w").write(text)
print(text)

### Table 3 (revised): detection rate (>=1 face, conf > 0.9), k/n (%), 95% Wilson CI

| Condition | Detector | sharp | low | medium | high | very_high |
|---|---|---|---|---|---|---|
| Sharp | RetinaFace | 156/156 (100.0%) [97.6–100.0] | – | – | – | – |
| Sharp | YuNet | 153/156 (98.1%) [94.5–99.3] | – | – | – | – |
| Blurred | RetinaFace | – | 156/156 (100.0%) [97.6–100.0] | 156/156 (100.0%) [97.6–100.0] | 97/156 (62.2%) [54.4–69.4] | 28/156 (17.9%) [12.7–24.7] |
| Blurred | YuNet | – | 151/156 (96.8%) [92.7–98.6] | 133/156 (85.3%) [78.8–90.0] | 32/156 (20.5%) [14.9–27.5] | 2/156 (1.3%) [0.4–4.6] |
| U-Net | RetinaFace | – | 156/156 (100.0%) [97.6–100.0] | 156/156 (100.0%) [97.6–100.0] | 156/156 (100.0%) [97.6–100.0] | 155/156 (99.4%) [96.5–99.9] |
| U-Net | YuNet | – | 154/156 (98.7%) [95.4–99.6] | 153/156 (98.1%) [94.5–99.3] | 152/156 (97.4%) [93.6–99.0] | 148/156 (94.9%) [90.2–97.4] |
| ViT | RetinaFace | – | 156/156 (100.0%) [97.6–100.0] | 156/156 (100.0%) [97.6–100.0] | 151/156 (

In [14]:
!cd /content && zip -q -r localization_eval.zip localization_eval
from google.colab import files
files.download("/content/localization_eval.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>